# GVC Measure 

Pulls OECD Trade-in-Value-Added (TiVa) API from each sector mapping to HS-6 Product Level in order to develop and create a GVC-intensity scoring measure.



In [7]:
# Pull BACII data from the database for 2017


import datetime
import os
import sys
import time 
import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine
from sqlalchemy import MetaData, Table, Column, Integer, String, DateTime, Float
from sqlalchemy.sql import select, and_, or_, not_
from sqlalchemy import func
from sqlalchemy import text
from sqlalchemy import inspect
from sqlalchemy import exc  
from sqlalchemy import event
from sqlalchemy.pool import Pool
import logging

logging.basicConfig()
logging.getLogger('sqlalchemy.engine').setLevel(logging.INFO)

## Data Loader for Project Codes

In [8]:
import pandas as pd

def load_product_codes():
    # Load HS17 product codes from data folder
    file_path = '/Users/apueelawekulom/Github/masters_thesis/data/BACI_HS17_V202601/product_codes_HS17_V202601.csv'
    
    try:
        df = pd.read_csv(file_path, dtype= 'str')
        # Fix leading zeros
        df['code'] = df['code'].str.zfill(6)  
        # Check all codes are exactly 6 digits
        lengths = df['code'].str.len().value_counts()
        print("Code length distribution:")
        print(lengths)
        
        
        print(f"Loaded {len(df)} product codes")
        print(df.head(20))
        return df
    
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        print("Available files:")
        import os
        data_dir = '/Users/apueelawekulom/Github/masters_thesis/data/'
        if os.path.exists(data_dir):
            print(os.listdir(data_dir))

if __name__ == "__main__":
    df_hs17 = load_product_codes()

Code length distribution:
code
6    5384
Name: count, dtype: int64
Loaded 5384 product codes
      code                                        description
0   010121           Horses: live, pure-bred breeding animals
1   010129  Horses: live, other than pure-bred breeding an...
2   010130                                        Asses: live
3   010190                            Mules and hinnies: live
4   010221           Cattle: live, pure-bred breeding animals
5   010229  Cattle: live, other than pure-bred breeding an...
6   010231          Buffalo: live, pure-bred breeding animals
7   010239  Buffalo: live, other than pure-bred breeding a...
8   010290  Bovine animals: live, other than cattle and bu...
9   010310            Swine: live, pure-bred breeding animals
10  010391  Swine: live, other than pure-bred breeding ani...
11  010392  Swine: live, other than pure-bred breeding ani...
12  010410                                        Sheep: live
13  010420                             

## HS Chapter, Subheadings, Descriptions

In [11]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import re

BASE    = "https://wits.worldbank.org/API/V1/wits/datasource/trn"
NS      = {"wits": "http://wits.worldbank.org"}
DATE_RE = re.compile(r"^\(([^)]*)\)\s*-?\s*")

def parse_description(raw: str) -> tuple[str, str]:
    m = DATE_RE.match(raw)
    if m:
        return m.group(1), raw[m.end():].strip()
    return "all", raw

# ── Layer 0 Chapter Prior (full HS6 universe, all 5 classes active) ───────────
# Single class per chapter range. Labels match plan §2.6 exactly.
# Class 5 now active — final goods scored, not gated out.

CLASS_NAMES = {
    1: "Raw/Primary",
    2: "Processed Material",
    3: "Component/Part",
    4: "Intermediate Assembly/Capital",
    5: "Final Good",
}

CHAPTER_PRIOR = [
    ( 1, 27, 1),   # live animals, food, raw materials, minerals, fuels
    (28, 40, 2),   # chemicals, plastics, rubber
    (41, 60, 3),   # leather, wood, paper, textiles (fabrics/yarns)
    (61, 63, 5),   # apparel and clothing accessories
    (64, 67, 5),   # footwear, headgear
    (68, 83, 3),   # stone, glass, ceramics, base metals, metal articles
    (84, 92, 4),   # machinery, electrical equipment, instruments
    (93, 93, 4),   # arms and ammunition
    (94, 96, 5),   # furniture, toys, miscellaneous manufactures
    (97, 97, 5),   # art, antiques
]

chapter_prior = pd.DataFrame([
    {
        "chapter":     str(ch).zfill(2),
        "prior_class": cls,
        "prior_label": CLASS_NAMES[cls],
    }
    for lo, hi, cls in CHAPTER_PRIOR
    for ch in range(lo, hi + 1)
])

print("Chapter prior class distribution:")
print(chapter_prior.groupby(["prior_class", "prior_label"])["chapter"]
      .count().rename("n_chapters").to_string())

# ── Source 1: GitHub CSV → chapter and heading text ───────────────────────────
print("\nFetching GitHub HS hierarchy CSV...")
gh = pd.read_csv(
    "https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv",
    dtype=str
)
gh["level"] = gh["level"].str.strip()

chapter_text = (gh[gh["level"] == "2"]
    .rename(columns={"hscode": "chapter", "description": "chapter_text"})
    [["chapter", "chapter_text"]])

heading_text = (gh[gh["level"] == "4"]
    .rename(columns={"hscode": "heading", "description": "heading_text"})
    [["heading", "heading_text"]])

print(f"  Chapters: {len(chapter_text)}, Headings: {len(heading_text)}")

# ── Source 2: WITS TRAINS → HS6 subheading text ──────────────────────────────
print("Fetching WITS HS6 subheadings...")
resp = requests.get(f"{BASE}/product/all", timeout=60)
resp.raise_for_status()
root = ET.fromstring(resp.content)

rows = []
for prod in root.findall(".//wits:product", NS):
    code    = prod.get("productcode", "").strip()
    isgroup = prod.get("isgroup", "Yes")
    raw     = prod.findtext("wits:productdescription",
                            default="", namespaces=NS).strip()
    if isgroup == "No" and len(code) == 6 and code.isdigit():
        year_range, desc = parse_description(raw)
        rows.append({"subheading":      code.zfill(6),
                     "subheading_text": desc,
                     "year_range":      year_range})

df_sub = pd.DataFrame(rows)
print(f"  Subheadings: {len(df_sub):,}")

# ── Assemble Layer 0 ──────────────────────────────────────────────────────────
df_sub["chapter"] = df_sub["subheading"].str[:2]
df_sub["heading"]  = df_sub["subheading"].str[:4]

df_layer0 = (df_sub
    .merge(chapter_text,  on="chapter", how="left")
    .merge(heading_text,   on="heading",  how="left")
    .merge(chapter_prior,  on="chapter",  how="left")
    [["chapter", "chapter_text",
      "heading",  "heading_text",
      "subheading", "subheading_text",
      "year_range",
      "prior_class", "prior_label"]])

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"\nRows:                  {len(df_layer0):,}")
print(f"Missing chapter text:  {df_layer0['chapter_text'].isna().sum()}")
print(f"Missing heading text:  {df_layer0['heading_text'].isna().sum()}")
print(f"Unassigned prior:      {df_layer0['prior_class'].isna().sum()}")
print(f"\nPrior class distribution (products):")
print(df_layer0.groupby(["prior_class","prior_label"], dropna=False)["subheading"]
      .count().rename("n_products").to_string())

# ── HS17 heading coverage check ───────────────────────────────────────────────
hs17_headings  = df_hs17["code"].str[:4].unique()
github_headings = set(heading_text["heading"].values)
missing_hd = [h for h in hs17_headings if h not in github_headings]
print(f"\nHS17 headings not in GitHub CSV: {len(missing_hd)}")
print(sorted(missing_hd))

df_layer0.head(10)

Chapter prior class distribution:
prior_class  prior_label                  
1            Raw/Primary                      27
2            Processed Material               13
3            Component/Part                   36
4            Intermediate Assembly/Capital    10
5            Final Good                       11

Fetching GitHub HS hierarchy CSV...
  Chapters: 97, Headings: 1229
Fetching WITS HS6 subheadings...
  Subheadings: 6,882

Rows:                  6,882
Missing chapter text:  0
Missing heading text:  89
Unassigned prior:      0

Prior class distribution (products):
prior_class  prior_label                  
1            Raw/Primary                      1348
2            Processed Material               1358
3            Component/Part                   2099
4            Intermediate Assembly/Capital    1497
5            Final Good                        580

HS17 headings not in GitHub CSV: 2
['8107', '8803']


,chapter,chapter_text,heading,heading_text,subheading,subheading_text,year_range,prior_class,prior_label
0,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010110,010110 -- (2002-2011) - Pure-bred breeding ani...,all,1,Raw/Primary
1,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010111,010111 -- (-2001) -- Pure-bred breeding animals,all,1,Raw/Primary
2,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010119,010119 -- (-2001) -- Other,all,1,Raw/Primary
3,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010120,"010120 -- (-2001) - Asses, mules and hinnies",all,1,Raw/Primary
4,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010121,010121 -- (2012-) -- Pure-bred breeding animals,all,1,Raw/Primary
5,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010129,010129 -- (2012-) -- Other,all,1,Raw/Primary
6,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010130,010130 -- (2012-) - Asses,all,1,Raw/Primary
7,01,Animals; live,0101,"Horses, asses, mules and hinnies; live",010190,010190 -- (2002-) - Other,all,1,Raw/Primary
8,01,Animals; live,0102,Bovine animals; live,010210,010210 -- (-2011) - Pure-bred breeding animals,all,1,Raw/Primary
9,01,Animals; live,0102,Bovine animals; live,010221,010221 -- (2012-) -- Pure-bred breeding animals,all,1,Raw/Primary
